In [ ]:
import pygame
import random

# Inicializa o Pygame
pygame.init()

# Configurações da Tela
LARGURA, ALTURA = 400, 400
TELA = pygame.display.set_mode((LARGURA, ALTURA))
pygame.display.set_caption("Quebra-cabeça - Pygame")

# Cores
BRANCO = (255, 255, 255)
PRETO = (0, 0, 0)
CINZA_CLARO = (200, 200, 200)
AZUL = (50, 150, 250)

# Configurações do Grid
TAMANHO_GRADE = 4
TAMANHO_PECA = LARGURA // TAMANHO_GRADE
FONTE = pygame.font.SysFont(None, 60)

def desenhar_grade(grade):
    """Desenha as peças na tela."""
    TELA.fill(PRETO)
    for linha in range(TAMANHO_GRADE):
        for coluna in range(TAMANHO_GRADE):
            valor = grade[linha][coluna]
            if valor != 0:
                # Desenha a peça
                pygame.draw.rect(
                    TELA, AZUL, 
                    (coluna * TAMANHO_PECA + 2, linha * TAMANHO_PECA + 2, 
                     TAMANHO_PECA - 4, TAMANHO_PECA - 4)
                )
                # Desenha o número
                texto = FONTE.render(str(valor), True, BRANCO)
                texto_retangulo = texto.get_rect(
                    center=(coluna * TAMANHO_PECA + TAMANHO_PECA // 2, 
                            linha * TAMANHO_PECA + TAMANHO_PECA // 2)
                )
                TELA.blit(texto, texto_retangulo)

def obter_posicao_vazio(grade):
    """Encontra a posição do espaço vazio (representado por 0)."""
    for l in range(TAMANHO_GRADE):
        for c in range(TAMANHO_GRADE):
            if grade[l][c] == 0:
                return l, c

def mover_peca(grade, clique_l, clique_c):
    """Move a peça clicada para o espaço vazio se for adjacente."""
    vazio_l, vazio_c = obter_posicao_vazio(grade)
    
    # Verifica se a peça clicada está adjacente ao espaço vazio
    if (abs(clique_l - vazio_l) == 1 and clique_c == vazio_c) or \
       (abs(clique_c - vazio_c) == 1 and clique_l == vazio_l):
        grade[vazio_l][vazio_c], grade[clique_l][clique_c] = grade[clique_l][clique_c], grade[vazio_l][vazio_c]
        return True
    return False

def embaralhar_grade():
    """Gera uma grade embaralhada garantindo que tenha solução."""
    grade = [[(i * TAMANHO_GRADE + j + 1) % (TAMANHO_GRADE * TAMANHO_GRADE) 
              for j in range(TAMANHO_GRADE)] for i in range(TAMANHO_GRADE)]
    
    # Simula movimentos aleatórios para criar um estado embaralhado solucionável
    movimentos = [(0, 1), (0, -1), (1, 0), (-1, 0)]
    vazio_l, vazio_c = TAMANHO_GRADE - 1, TAMANHO_GRADE - 1
    
    for _ in range(100):
        random.shuffle(movimentos)
        for dl, dc in movimentos:
            novo_l, novo_c = vazio_l + dl, vazio_c + dc
            if 0 <= novo_l < TAMANHO_GRADE and 0 <= novo_c < TAMANHO_GRADE:
                grade[vazio_l][vazio_c] = grade[novo_l][novo_c]
                grade[novo_l][novo_c] = 0
                vazio_l, vazio_c = novo_l, novo_c
                break
    return grade

def verificar_vitoria(grade):
    """Verifica se o quebra-cabeça foi resolvido."""
    valor_esperado = 1
    for l in range(TAMANHO_GRADE):
        for c in range(TAMANHO_GRADE):
            if l == TAMANHO_GRADE - 1 and c == TAMANHO_GRADE - 1:
                return grade[l][c] == 0
            if grade[l][c] != valor_esperado:
                return False
            valor_esperado += 1
    return True

def main():
    grade = embaralhar_grade()
    rodando = True
    jogando = True
    movimentos_totais = 0
    mensagem_vitoria = False

    while rodando:
        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                rodando = False
            
            if evento.type == pygame.MOUSEBUTTONDOWN and jogando:
                x, y = pygame.mouse.get_pos()
                coluna = x // TAMANHO_PECA
                linha = y // TAMANHO_PECA
                
                if mover_peca(grade, linha, coluna):
                    movimentos_totais += 1
                    print(f"Movimentos: {movimentos_totais}")
                    
                    if verificar_vitoria(grade):
                        print("Parabéns! Você resolveu o quebra-cabeça!")
                        jogando = False
                        mensagem_vitoria = True

        desenhar_grade(grade)

        if mensagem_vitoria:
            # Exibe mensagem de vitória por cima da tela
            superficie_vitoria = FONTE.render("Vitória!", True, BRANCO)
            retangulo_vitoria = superficie_vitoria.get_rect(center=(LARGURA//2, ALTURA//2))
            pygame.draw.rect(TELA, PRETO, retangulo_vitoria.inflate(20, 20))
            TELA.blit(superficie_vitoria, retangulo_vitoria)

        pygame.display.flip()

    pygame.quit()

if __name__ == "__main__":
    main()